# Raster calculations for Semantic segmentation modelling
Code based on given code from Gitbook > Training data > Vector to masks  

Chose to go with OSM building footprints, even though they are not as detailed and precise as the swissBUILDINGS3D dataset. Maybe post-checks bc of that ????

In [2]:
# loading libraries
from pathlib import Path
import geoai
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import numpy as np

In [5]:
## Loading raster data
PROJECT_DIR = Path.cwd().parent
print(PROJECT_DIR.resolve)

raster_dir= PROJECT_DIR / "data" / "raw" / "raster" / "vexcel" / "luzern-2025_epsg2056.tif"

with rasterio.open(raster_dir) as src:
    print("CRS:", src.crs)
    print("Size:", src.width, src.height)
    print("Bands:", src.count)


<bound method Path.resolve of WindowsPath('c:/Users/Maja/Documents/04-Studium/SDS320/project')>
CRS: EPSG:2056
Size: 13334 13334
Bands: 4


In [6]:
# Loading vector data
vector_path = PROJECT_DIR / "data" / "raw" / "vector" / "osm" / "OSM_buildings.geojson"

gdf_buildings = gpd.read_file(vector_path)

print("Vector CRS:", gdf_buildings.crs)
print(gdf_buildings.geom_type.unique())
print(len(gdf_buildings))
print(gdf_buildings.head())

Vector CRS: EPSG:2056
['Polygon']
765
      building                                           geometry
0  residential  POLYGON ((2666198.46 1209935.72, 2666187.977 1...
1   apartments  POLYGON ((2666617 1209968.564, 2666623.853 120...
2   apartments  POLYGON ((2666045.503 1209980.109, 2666036.703...
3          yes  POLYGON ((2666818.372 1209990.739, 2666805.363...
4   apartments  POLYGON ((2666008.969 1209983.273, 2666000.285...


c:\Users\Maja\programming\anaconda3\envs\sds320_v2\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [7]:
## Verify correct CRS
if gdf_buildings.crs != src.crs:
    gdf_buildings = gdf_buildings.to_crs(src.crs)

print("Reprojected vector CRS:", gdf_buildings.crs)

Reprojected vector CRS: EPSG:2056


#### B) Rasterise Vector polygons 
First, we need to rasterize the building vectors to a building mask (0= no building, 1 = building).

In [ ]:
mask_path = PROJECT_DIR / "data" / "processed" / "building_mask.tif"

geoai.vector_to_raster(
    gdf_buildings,
    mask_path,
    reference_raster=raster_dir,
)
# class values should be 1 for buildings and 0 for background
# how can i check this ?????

array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(13334, 13334), dtype=uint8)

In [ ]:
geoai.view_image(mask_path, figsize=(18,10))
# Doesn't work, kernel always crashes....

## how can i control edge cases ???


NameError: name 'geoai' is not defined

#### Vegetation Mask
Create a vegetation mask based on a NDVI threshold -> how large should this threshold be ????

In [6]:
## start with a NDVI threshold of 0.2 (starting point)
#  for denser vegetation, we can increase the threshold to 0.3 or 0.4) 
with rasterio.open(raster_dir) as src:
    print(src.count)
    print(src.descriptions)

4
('blue', 'green', 'red', 'nir')


In [10]:
mask_path = PROJECT_DIR / "data" / "processed"

## Compute NDVI
with rasterio.open(raster_dir) as src:

    red = src.read(1).astype(np.float32)   # change band number if needed
    nir = src.read(2).astype(np.float32)   # change band number if needed

    ndvi = (nir - red) / (nir + red + 1e-10)

    profile = src.profile.copy()

## Create the vegetation mask based on the NDVI threshold
ndvi_threshold = 0.2
vegetation_mask = (ndvi >= ndvi_threshold).astype(np.uint8)

## Save the mask
profile.update(
    dtype=rasterio.uint8,
    count=1,
    nodata=0
)

out_path = mask_path / "vegetation_mask.tif"

with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(vegetation_mask, 1)

print(f"Mask saved to: {out_path}")

Mask saved to: c:\Users\Maja\Documents\04-Studium\SDS320\project\data\processed\vegetation_mask.tif
